In [2]:
import os
import scanpy as sc
import os
from scipy import io
save_dir = '/stornext/General/data/user_managed/grpu_mritchie_1/reza/projects/mat_seq/RaCHseq_AML/cellranger-arc_outputs/aggr_outputs/AML_BMMC_38_Samples_no_secondary_analysis/subset_screening_all_aml/'

# Change the working directory
new_dir = "/stornext/General/data/user_managed/grpu_mritchie_1/reza/projects/mat_seq/RaCHseq_AML/seurat_outputs/AML_BMMC_ALL/Jupyter"
os.chdir(new_dir)

Current working directory: /stornext/Home/data/allstaff/g/ghamsari.r/R_scripts_year2/Monosomy_All_v1
Updated working directory: /stornext/General/data/user_managed/grpu_mritchie_1/reza/projects/mat_seq/RaCHseq_AML/seurat_outputs/AML_BMMC_ALL/Jupyter


In [3]:
adata = sc.read_10x_mtx(
    '/stornext/General/data/user_managed/grpu_mritchie_1/reza/projects/mat_seq/RaCHseq_AML/cellranger-arc_outputs/aggr_outputs/AML_BMMC_38_Samples_no_secondary_analysis/outs/filtered_feature_bc_matrix',  # the directory with the `.mtx` file                # use gene symbols for the variable names (variables-axis index)
    gex_only=False)   
#<314235x221141 sparse matrix of type '<class 'numpy.float32'>'
# with 3034982842 stored elements in Compressed Sparse Row format>

In [42]:
#subsampling the samples of interest, here I used 25 patients and 4 healthy donor as the order of cellranger-arc -aggr output.

# Start with numbers 1 through 25
AML_samples = list(range(1, 26))

# List of additional numbers to add BMMC_S1_D1, BMMC_S2_D1, BMMC_S4_D1 and BMMC_S4_D8
healthy_Samples = [26, 29, 36, 37]

# Extend the original list with the additional numbers
AML_samples.extend(healthy_Samples)

desired_suffixes = ["-" + str(i) for i in AML_samples] # generates list ['-1', '-2', '-3', ..., '-10']

mask = [any(barcode.endswith(suffix) for suffix in desired_suffixes) for barcode in adata.obs_names]
adata_subset_sample = adata[mask]
len(list(adata_subset.obs_names)) #232173


232173

In this code, adata_subsets is a dictionary where the keys are the unique feature types and the values are the AnnData objects corresponding to that feature type.

In [44]:
feature_types = adata.var["feature_types"].unique()
adata_subsets = {}

for feature_type in feature_types:
    mask = adata.var["feature_types"] == feature_type
    adata_subsets[feature_type] = adata_subset_sample[:, mask]
ATAC = adata_subsets['Peaks']
GEX =adata_subsets['Gene Expression']

In [47]:
#saving ATAC data
save_dir = '/stornext/General/data/user_managed/grpu_mritchie_1/reza/projects/mat_seq/RaCHseq_AML/cellranger-arc_outputs/aggr_outputs/AML_BMMC_38_Samples_no_secondary_analysis/subset_all_aml_atac_4healthy/'
!mkdir -p {save_dir}
adata_temp = ATAC
# Construct the complete file path for barcodes.tsv
barcode_path = os.path.join(save_dir, 'barcodes.tsv')

with open(barcode_path, 'w') as f:
    for item in adata_temp.obs_names:
        f.write(item + '\n')

# Construct the complete file path for features.tsv
features_path = os.path.join(save_dir, 'features.tsv')

with open(features_path, 'w') as f:
    for item in ['\t'.join([x, x, 'Peaks']) for x in adata_temp.var_names]:
        f.write(item + '\n')

# Construct the complete file path for matrix.mtx
matrix_path = os.path.join(save_dir, 'matrix.mtx')

# Use the io.mmwrite function from scipy to write the matrix in Matrix Market format
io.mmwrite(matrix_path, adata_temp.X.T)

# Compress the contents of the save_dir directory using gzip
!gzip -r {save_dir}/*

#saving RNA data

save_dir = '/stornext/General/data/user_managed/grpu_mritchie_1/reza/projects/mat_seq/RaCHseq_AML/cellranger-arc_outputs/aggr_outputs/AML_BMMC_38_Samples_no_secondary_analysis/subset_all_aml_gex_4healthy/'
!mkdir -p {save_dir}/
adata_temp = GEX
# Construct the complete file path for barcodes.tsv
barcode_path = os.path.join(save_dir, 'barcodes.tsv')

with open(barcode_path, 'w') as f:
    for item in adata_temp.obs_names:
        f.write(item + '\n')

# Construct the complete file path for features.tsv
features_path = os.path.join(save_dir, 'features.tsv')

with open(features_path, 'w') as f:
    for item in ['\t'.join([x, x, 'Gene Expression']) for x in adata_temp.var_names]:
        f.write(item + '\n')

# Construct the complete file path for matrix.mtx
matrix_path = os.path.join(save_dir, 'matrix.mtx')

# Use the io.mmwrite function from scipy to write the matrix in Matrix Market format
io.mmwrite(matrix_path, adata_temp.X.T)

# Compress the contents of the save_dir directory using gzip
!gzip -r {save_dir}/*
